# Aditivo de casos visuais: preparação offline

Seis tarefas preparadas para pesquisa local. Este notebook confere arquivos e prepara doze entradas com/sem imagem. Não instala, baixa, faz chamadas de API ou executa modelos. As células não leem referências de avaliação. O contrato é próprio deste aditivo; não substitui o runner original.

Aponte INPUTS para model-input. Para rever a referência proposta, use uma sessão de avaliação separada. Nenhum resultado experimental está incluído.

Cada condição futura deve usar uma chamada/sessão independente. Mostrar a imagem em uma conversa e depois retirá-la não cria um controle sem imagem. As referências continuam pendentes de revisão humana.


In [ ]:
from pathlib import Path
import json, hashlib

# Aponte INPUTS apenas para a pasta model-input deste aditivo.
INPUTS = Path("model-input")
assert INPUTS.is_dir(), "Disponibilize apenas a pasta model-input ao lado deste notebook e ajuste INPUTS."
base = INPUTS.resolve()
cases = [json.loads(p.read_text(encoding="utf-8")) for p in sorted(INPUTS.glob("ADD[0-9][0-9].json"))]
assert cases, "Nenhum caso neutro disponível."
seen = set()
for case in cases:
    assert case["case_id"] not in seen, "ID duplicado"
    seen.add(case["case_id"])
    image = (INPUTS / case["image"]).resolve()
    assert image.is_relative_to(base) and image.is_file(), "Estímulo ausente ou caminho fora da pasta neutra"
    assert hashlib.sha256(image.read_bytes()).hexdigest() == case["image_sha256"], "Hash de imagem divergente"
    assert case["output_contract"]["without_image"] == "[],undetermined(visual_facts,doc1)"
print(f"{len(cases)} tarefas neutras carregadas; arquivos e hashes conferidos. Nenhum modelo chamado.")


In [ ]:
def compose_prompt(case):
    return "\n\n".join([
        f"CASO: {case['case_id']}.",
        "Use somente o contexto, o entimema, a regra e o contrato abaixo e, na condição with_image, os pixels fornecidos. Não use fontes externas, metadados descritivos ou outras conversas.",
        "CONTEXTO\n" + case["context"],
        "ENTIMEMA\n" + case["enthymeme"],
        "REGRA FORNECIDA\n" + case["provided_rule"],
        "CONTRATO DE SAÍDA\n" + json.dumps(case["output_contract"], ensure_ascii=False),
        "VOCABULÁRIO\n" + json.dumps(case.get("vocabulary", {}), ensure_ascii=False),
    ])

prepared_inputs = []
for case in cases:
    for condition in ["with_image", "without_image"]:
        prepared_inputs.append({
            "record_type": "prepared_input", "status": "prepared_not_executed",
            "case_id": case["case_id"], "condition": condition, "repeat": 1,
            "prompt": compose_prompt(case),
            "image": case["image"] if condition == "with_image" else None,
            "image_sha256": case["image_sha256"] if condition == "with_image" else None,
        })
print(f"{len(prepared_inputs)} entradas preparadas em memória. Isso não é execução de modelo.")


In [ ]:
# Exibe apenas estímulos neutros; não carrega referências, fontes ou gabaritos.
from IPython.display import display, Image
selected = cases[0]
print(selected["case_id"])
display(Image(filename=str(INPUTS / selected["image"])))
print(compose_prompt(selected))


As entradas estão em `prepared_inputs`, com status `prepared_not_executed`. Também há uma cópia neutra em `model-input/prepared-inputs.jsonl`. Não confundir estas entradas com respostas candidatas. Uma execução futura precisa de autorização própria, configuração e registros novos.
